# Text-to-score transformation of Free-text Notes

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import gzip

# Import necessary global variables
from config.config import *

### Prepare MIMIC notes data
Each note in the MIMIC 'noteevents' table is linked to its corresponding patient ('subject_id') and hospital admission ('hadm_id').

We need to associate each MIMIC note to its corresponding patient ICU stay id ('icustay_id'), so that they can be tracked as vital signs and lab values per each patient stay.

In [ ]:
# NOTE:

# To obtain the MIMIC-III subset, we run this notebook once with MIMIC-III data

# To obtain the MIMIC-III+IV subset, we need to run this notebook twice:
# 1st: MIMIC-III subset data (it contains patients in MIMIC-III not present in MIMIC-IV)
# 2nd: MIMIC-IV data
# Finally, to create the final MIMIC-III+IV dataframe we need to combine the resulting outputs of MIMIC-III subset and MIMIC-IV into a single file

In [ ]:
path_mimic = PATH_MIMIC

csv_icustays = "ICUSTAYS.csv.gz"
csv_noteevents = "NOTEEVENTS.csv.gz"

In [ ]:
# Check 'icustays' table
file_icustays = path_mimic + csv_icustays
with gzip.open(file_icustays, "rt") as file:
    df_icustays = pd.read_csv(file)

# Drop 'hadm_id', 'icustay_id', 'intime' and 'outtime' containing NaN
df_icustays = df_icustays.dropna(subset=["hadm_id"])
df_icustays = df_icustays.dropna(subset=["icustay_id"])
df_icustays = df_icustays.dropna(subset=["intime"])
df_icustays = df_icustays.dropna(subset=["outtime"])

df_icustays.head()

In [ ]:
# Check 'noteevents' table
file_noteevents = path_mimic + csv_noteevents
with gzip.open(file_noteevents, "rt") as file:
    df_noteevents = pd.read_csv(file)

# Drop 'hadm_id' and 'charttime' containing NaN
df_noteevents = df_noteevents.dropna(subset=["hadm_id"])
df_noteevents = df_noteevents.dropna(subset=["charttime"])

df_noteevents.head()

In [ ]:
# Get the 'icustay_id' associated to each note

# Convert 'charttime' column to datetime objects
df_noteevents['charttime'] = pd.to_datetime(df_noteevents['charttime'])

# Convert 'intime' and 'outtime' columns in df_icustays to datetime
df_icustays['intime'] = pd.to_datetime(df_icustays['intime'])
df_icustays['outtime'] = pd.to_datetime(df_icustays['outtime'])

# Merge df_noteevents with df_icustays based on 'hadm_id'
merged_df = pd.merge(df_noteevents, df_icustays, on='hadm_id', how='inner')

# Filter rows based on charttime and ICU stay duration
def filter_icustays(row):
    return (row['charttime'] >= row['intime']) and (row['charttime'] <= row['outtime'])

# Apply the filter to the merged DataFrame
df_noteevents_icu = merged_df[merged_df.apply(filter_icustays, axis=1)]

# The DataFrame now contains the 'icustay_id' associated with each note
df_noteevents_icu.head()

In [ ]:
# Create a new DataFrame containing only the relevant columns
df_noteevents_clean = df_noteevents_icu[["icustay_id", "subject_id_x", "hadm_id", "charttime", "text"]]
# Rename 'subject_id_x' column to 'subject_id'
df_noteevents_clean = df_noteevents_clean.rename(columns={'subject_id_x': 'subject_id'})

# Remove line break special character '\n' from the 'text' column
df_noteevents_clean["text"] = df_noteevents_clean["text"].str.replace('\n', '', regex=True)
# Remove date and time from the 'text' column
df_noteevents_clean['text'] = df_noteevents_clean['text'].str.replace(r'\[\*\*\d{4}-\d{1,2}-\d{1,2}\*\*\] \d{1,2}:\d{1,2} [APapMm]{2}', '', regex=True)

df_noteevents_clean.head()

In [ ]:
# Save the new dataframe into a pickle file
# df_noteevents_clean.to_pickle(f"data/df_noteevents_clean.pkl")

### Select text encoder and text-to-score transformation

#### Text encoder (A): ISeeU2

In [ ]:
if TEXT_ENCODER == "ISeeU2":    
    import pickle
    import pkg_resources
    import nltk
    nltk.download('stopwords')
    from nltk.corpus import stopwords
    import numpy as np
    from scipy.ndimage import convolve1d
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    import matplotlib as mpl

    import tensorflow
    import keras
    from tensorflow.keras.models import Model, load_model
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.compat.v1.keras.backend import get_session
    tensorflow.compat.v1.disable_v2_behavior()

    import tensorflow as tf
    tf.compat.v1.enable_eager_execution()

In [ ]:
if TEXT_ENCODER == "ISeeU2":
    # Define the ISeeU2 class
    class ISeeU2:
        __version__ = "0.1.0"

        def __init__(self):
            model_file = PATH_TEXT_ENCODER + 'models/kfold2_best_F1.h5'
            print(f"****{model_file}*****")
            self._model = load_model(model_file)

            self._stops = set(stopwords.words("english"))
            tokenizer_file = PATH_TEXT_ENCODER + 'res/tokenizer.pkl'

            with open(tokenizer_file, 'rb') as t:
                self._tokenizer = pickle.load(t)
            self._sequence_max_length = 512

            word_index = self._tokenizer.word_index
            self._reverse_word_index = dict(
                [(value, key) for (key, value) in word_index.items()])

        def preprocess_notes(self, notes):
            def f(x): return ' '.join(
                [item for item in x.split() if item not in self._stops])

            notes.loc[:, 'text'] = notes['text'].apply(f)

            notes_sequences = self._tokenizer.texts_to_sequences(
                notes['text'].values)
            padded_notes_sequences = pad_sequences(notes_sequences,
                                                self._sequence_max_length,
                                                truncating='pre')
            return padded_notes_sequences

        def predict(self, patient_sequences):
            mortality = self._model.predict(patient_sequences)

            return mortality

In [ ]:
if TEXT_ENCODER == "ISeeU2":
    predictor = ISeeU2()
    patients_notes = predictor.preprocess_notes(df_noteevents_clean)

    df_noteevents_and_mortality = pd.DataFrame(df_noteevents_clean)
    df_noteevents_and_mortality["mortality_risk_prediction"] = predictor.predict(patients_notes)
    df_noteevents_and_mortality

#### Text encoder (B): CORe

In [ ]:
if TEXT_ENCODER == "CORe": 

    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    import torch

    # Check if GPU is available
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print("Device:", device)

    # Initialise the tokenizer and model on the GPU
    tokenizer = AutoTokenizer.from_pretrained("bvanaken/CORe-clinical-mortality-prediction")
    model = AutoModelForSequenceClassification.from_pretrained("bvanaken/CORe-clinical-mortality-prediction")
    model.to(device)

    # Predict mortality risk for each text
    def predict_mortality_risk(text):
        tokenized_input = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)  # Move input to GPU
        output = model(**tokenized_input)
        predictions = torch.softmax(output.logits.detach(), dim=1)
        mortality_risk_prediction = predictions[0][1].item()
        
        return mortality_risk_prediction

    # Apply the function to each row in the DataFrame
    df_noteevents_and_mortality = pd.DataFrame(df_noteevents_clean)
    df_noteevents_and_mortality['mortality_risk_prediction'] = df_noteevents_and_mortality['text'].apply(predict_mortality_risk)
    df_noteevents_and_mortality

### Save the new DataFrame containing the textual transformation as mortality scores

In [ ]:
# Save the new dataframe into a pickle file
df_noteevents_and_mortality.to_pickle(OUTPUT_PATH_NOTES_SCORES)